# Lab — Optimising Calculations with NumPy Arrays (Solution)


## 1) Setup & Environment Check

Make sure NumPy is installed and importable.


In [ ]:
# NumPy availability check
try:
    import numpy as np
    print("NumPy version:", np.__version__)
except ImportError:
    raise ImportError(
        "NumPy is not installed. Install it with one of the following:\n"
        "  - pip install numpy\n"
        "  - conda install numpy"
    )


## 2) Create a Dataset to Work With

We’ll generate a **large numeric dataset** so performance differences are noticeable.

Your job:
1. Create a 1D NumPy array called `values` with **at least 1,000,000** random floats.
2. Confirm the array’s shape and dtype.


In [ ]:
# Create a large array of random floats
# Using a seed makes results reproducible.
import numpy as np

rng = np.random.default_rng(seed=42)
values = rng.random(size=5_000_000)  # 5 million floats in [0, 1)

values[:5]


In [ ]:
print("Shape:", values.shape)
print("Dtype:", values.dtype)
print("Min/Max:", values.min(), values.max())


## 3) Define the Same Calculation Two Ways

We’ll compute **min-max normalisation**:

\[ \text{norm}(x) = \frac{x - \min(x)}{\max(x) - \min(x)} \]

### A) Loop-Based Version (Python)
Write a function `normalize_loop(values)` that:
- Iterates through each element
- Computes the normalised value
- Returns a **Python list** (or a NumPy array — your choice, but be consistent)

### B) Vectorised Version (NumPy)
Write a function `normalize_vectorized(values)` that:
- Uses NumPy operations (no explicit Python loops)
- Returns a NumPy array


In [ ]:
import time

def normalize_loop(values):
    """Normalise using Python loops (intentionally slower).

    Returns a new list of floats scaled to [0, 1].
    """
    vmin = float(min(values))
    vmax = float(max(values))
    denom = (vmax - vmin) if (vmax - vmin) != 0 else 1.0

    out = []
    for v in values:
        out.append((float(v) - vmin) / denom)
    return out


In [ ]:
def normalize_vectorized(values):
    """Normalise using NumPy vectorised operations.

    Returns a NumPy array scaled to [0, 1].
    """
    vmin = values.min()
    vmax = values.max()
    denom = (vmax - vmin) if (vmax - vmin) != 0 else 1.0
    return (values - vmin) / denom


## 4) Sanity Check: Do Both Methods Match?

Run both functions on a **small sample** first.  
Your outputs should be numerically very close.


In [ ]:
# Quick correctness check on a smaller slice
sample = values[:10_000]

out_loop = normalize_loop(sample)              # list
out_vec = normalize_vectorized(sample)         # ndarray

# Convert list -> ndarray for comparison
out_loop_arr = np.array(out_loop, dtype=float)

print("Same shape:", out_loop_arr.shape == out_vec.shape)
print("All close:", np.allclose(out_loop_arr, out_vec, rtol=1e-9, atol=1e-12))


## 5) Time the Two Approaches

Use `time.perf_counter()` to measure runtime.

Guidelines:
- Time each method at least **2 times** and note the range.
- Use the **same input** for both methods.
- Keep the output assignment so work isn’t optimised away.


In [ ]:
from time import perf_counter

# Time loop-based approach (note: returns a Python list)
start = perf_counter()
out_loop_full = normalize_loop(values)
end = perf_counter()
loop_seconds = end - start

# Time vectorized approach (returns a NumPy array)
start = perf_counter()
out_vec_full = normalize_vectorized(values)
end = perf_counter()
vec_seconds = end - start

speedup = loop_seconds / vec_seconds if vec_seconds != 0 else float("inf")

print(f"Loop time:       {loop_seconds:.3f}s")
print(f"Vectorized time: {vec_seconds:.3f}s")
print(f"Speedup:         {speedup:.1f}x")

# Optional: quick check the outputs match (convert loop output to ndarray)
out_loop_full_arr = np.array(out_loop_full, dtype=float)
print("Outputs match (allclose):", np.allclose(out_loop_full_arr, out_vec_full, rtol=1e-9, atol=1e-12))


## 6) Document Your Findings (Markdown)

In a markdown cell below, answer:
1. Which approach was faster? By roughly how much?
2. Why does NumPy’s vectorisation usually outperform loops?
3. What tradeoffs might exist (memory, readability, setup cost)?

*(Write your answers under “Results & Reflection”.)*


### Results & Reflection

In most environments, the vectorised NumPy version runs dramatically faster than the Python loop version.

Why:
- NumPy performs operations in optimized, compiled code under the hood.
- Vectorised operations reduce Python-level iteration overhead.
- Array operations are designed to work on entire blocks of values at once.

Takeaway:
When working with large numerical datasets, prefer NumPy arrays and vectorised operations to improve both performance and readability.


## 7) Optional Extension (If You Have Time)

Try one of the following:
- Compute a **z-score standardisation** instead of min-max normalisation
- Compare performance of `np.sum(values)` vs a Python loop sum
- Create a 2D array and perform a vectorised row-wise calculation


## Deliverables

Make sure your notebook includes:
- A working loop-based function and a vectorised function
- A correctness check (e.g., `np.allclose`)
- Timing results using `perf_counter`
- Written reflection in markdown

When you’re done, commit and push your work to GitHub.
